# 实验一 · 应用框架与运行时资源 —— 七步流程与资源生命周期

**所属**：《并行计算》第七章 · AscendCL 应用开发　|　**难度**：⭐⭐ 基础　|　**预计时长**：30–40 分钟

第六章的每一个实验都以核函数为中心，主机侧的代码只是把核函数送上设备的一段附属工作。从本实验开始，主机侧的代码成为讨论重点。

本实验从一个向量加法出发，写出第一个完整的 acl 应用，顺序与官方入门样例的七步一致：初始化与指定 Device、创建 Context 与 Stream、准备输入、调用一个 CANN 内置算子、取回结果、逆序释放资源、去初始化。程序本身不足两百行，它确立的资源配对规则与错误定位方法是后面各节反复使用的基础。

> **实验说明**
> 1. 本实验是第七章的起点，确立本章统一的编译、运行与排错流程。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**，建议在 CANNLab 云开发环境中运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. **源代码通过 `%%writefile` 分段写入同一个 `.cpp` 文件**——每段配一节讲解，最后写入 `main` 函数。本章的源文件是主机侧的 C++ 程序，编译器不再是 `bisheng` 而是 `g++`。
> 6. 本实验先用一条 `g++` 命令编译，随后再演示 CMake 工程方式，并对照二者的适用场景。
> 7. **本实验不做性能测量。** 环境规格查询、资源配对自检与错误码定位这三件事，是任何性能测量能够得出可信结论的前提，本实验先把它们做扎实。
> 8. 若环境检查未通过，请先重新运行第 4 节的环境准备单元格；仍不通过则在终端执行 `source $ASCEND_TOOLKIT_HOME/set_env.sh` 并重启内核。


## 🎯 学习目标

完成本实验后，学生应能够：

- 说明主机与设备各自拥有独立内存空间、主机与设备异步并行执行、Stream 内保序而 Stream 间并行这三条编程原则
- 说明 Device、Context、Stream、Task 四级抽象之间的从属关系，以及 Context 管理哪些资源、不管理哪些资源
- 说明线程与 Context 的关联规则，解释为何多数 Runtime 接口不带 Device ID 参数
- 区分默认 Context 与显式创建的 Context，说明二者各自的使用限制与适用场景
- 根据程序所包含的头文件确定需要链接的库文件，并说明为何不应把库名写死在编译命令里
- 用 `g++` 与 CMake 两种方式编译同一份 acl 源码，并说明何时才需要引入构建系统
- 编写资源配对完整、每一次接口调用都检查返回码的 acl 应用程序，并说出释放顺序的两条规则
- 使用 `aclGetRecentErrMsg` 获取接口调用失败时的错误描述，并按返回码的分类规则判断故障方向
- 用 `aclrtGetSocName` 与 `aclrtGetDeviceInfo` 查询本机规格，与第六章的硬件参照表相互印证


## 🗺️ 学习路径

1. **准备阶段**：理解主机与设备各自拥有独立的内存，且任务是异步执行的，因而同步必须显式写出
2. **概念建立**：Runtime 的四级抽象，以及 Device、Context、Stream 与主机线程之间的关联
3. **工程约束**：接口的命名规范，以及头文件与库文件之间的对应关系
4. **程序实现**：按七步框架写出一个完整的 acl 应用，理解每一步各自解决什么问题
5. **两种构建方式**：同一份源码分别用一条 `g++` 命令与一个 CMake 工程编译，理解工程规模变大之后为什么需要构建系统
6. **排错方法**：由返回码的分类判断故障方向，再用错误描述接口定位到具体的失败调用
7. **资源生命周期**：建立申请与释放成对出现的检查习惯，理解释放次序为什么不能颠倒


## 1. 背景与动机：主机与设备的分工

CANN 的编程模型建立在一对概念之上。

**主机**（Host）指 X86 服务器 CPU 或 ARM 服务器 CPU，它通过总线与一个或多个设备互联，利用设备提供的神经网络计算能力完成业务。**设备**（Device，通常也称 NPU）指安装了昇腾 AI 处理器的硬件，它通过总线与主机相连，提供神经网络等计算能力。总线可以是 PCIe，也可以是 HCCS（Huawei Cache Coherence System，华为缓存一致性系统）。

<img src="images/07.01_host_device.png" alt="主机与设备的关系" height="430">

图中有三点值得注意。第一，主机与设备各自拥有独立的互联控制器与内存，两者之间只通过总线相连。第二，设备内部除了负责矩阵与向量计算的 AI Core，还有 AI CPU、任务调度器以及其它加速器。第三，一台主机下可以挂接多个设备，图中层叠的虚线框表示的正是这一点。

第六章的全部内容位于图中 Device 一侧，发生在 AI Core 与设备内存之间。本章要处理的是这张图的其余部分：主机侧的内存、总线上的数据搬运，以及设备侧的任务调度器。

### 1.1 三条编程原则

**原则一：主机和设备各自拥有独立的内存空间。**

Runtime 分别提供了主机侧和设备侧的内存申请接口，以及主机与设备之间的内存复制接口。编程时需要区分主机和设备的内存申请，并显式调用内存复制接口，将数据从主机侧复制到设备侧，以使设备硬件加速器在访问本地内存时达到最佳性能。

换言之，设备内存中的数据必须由程序显式申请并显式搬入。本实验中，两个输入向量在主机内存里生成，必须由程序显式申请设备内存、显式拷贝过去；结果也必须显式拷贝回来。

**原则二：主机与设备之间采用异步并行的执行方式。**

主机将任务下发到设备后，不会等待设备任务执行完成就立即返回；设备随即开始调度并执行下发的任务；主机侧的 CPU 可以与设备侧的加速器并行工作。异步任务下发的接口通常会带有 Stream 参数，表示将任务下发到对应的 Stream 中执行。

当主机需要获取设备的计算结果时，必须发起显式的同步接口调用。同步接口会阻塞主机 CPU，直到设备侧任务执行完成才返回。

下图展示了一次任务下发与取回结果的完整过程。主机侧的应用调用下发接口把任务加入 Stream 对应的队列，设备侧的任务调度器从队列中取出任务、按类型分派给 CPU 执行器或 AI Core，执行完成后回填状态；主机侧再通过同步接口获取执行结果。

<img src="images/07.01_task_scheduling.png" alt="基于 Runtime 编程的典型执行流程" height="500">

图给出的是任务的流向，异步语义要由接口语义补足：步骤 1 把任务加入队列之后，下发接口即返回，主机侧的 CPU 可以继续工作，直到步骤 6 的同步接口才会阻塞。

**与第六章的对照**：这条原则在第六章针对核函数已经强调过——`<<<>>>` 下发之后必须 `aclrtSynchronizeStream` 才能确认执行完毕。本章把同一条规则推广到算子、模型与拷贝任务：凡是异步接口，返回成功都只表示下发成功。

**原则三：异步任务下发到 Stream 中，Stream 内的任务保序执行，Stream 间的任务并行执行。**

下图中主机侧顺序启动 Kernel1、Kernel2、Kernel3 三个任务。Kernel1 和 Kernel3 位于同一个 Stream 中，因此 Kernel3 需要等待 Kernel1 执行完毕才能开始执行；Kernel2 与另外两个不在同一个 Stream 中，因此可以与它们并行执行。

<img src="images/07.01_stream_parallel.png" alt="Stream 内保序与 Stream 间并行" width="800px">

图中以核函数下发语法 `KernelN<<<numBlockN, nullptr, streamN>>>` 举例，本实验用的是算子下发接口；两者的 `stream` 参数含义相同，图中的保序与并行关系同样成立。

需要强调的是，Stream 间能否真正并行取决于硬件资源。硬件资源充足时，不同 Stream 上的任务会被调度到不同的硬件资源上并行执行；硬件资源不足时，不同 Stream 上的任务串行执行。

三条原则在本实验中的落点如下：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 原则 | 本实验中的体现 |
| --- | --- |
| 独立内存空间 | `aclrtMalloc` 申请设备内存，`aclrtMemcpy` 显式搬运 |
| 异步并行执行 | 算子下发后必须 `aclrtSynchronizeStream` 才能取结果 |
| Stream 内保序、Stream 间并行 | 本实验只用一条 Stream，仅建立概念 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">原则</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验中的体现</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">独立内存空间</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMalloc</code> 申请设备内存，<code>aclrtMemcpy</code> 显式搬运</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步并行执行</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子下发后必须 <code>aclrtSynchronizeStream</code> 才能取结果</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Stream 内保序、Stream 间并行</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">本实验只用一条 Stream，仅建立概念</td>
</tr>
</tbody>
</table>


## 2. Runtime 的四级抽象与线程模型

### 2.1 四级抽象

Runtime 用四个概念描述设备上的执行环境，它们之间是层层嵌套的从属关系。

<img src="images/07.01_runtime_abstraction.png" alt="Device、Context、Stream、Task 的从属关系" width="550px">

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 定义 | 数量关系 |
| --- | --- | --- |
| Device | 对昇腾 AI 处理器所属设备的抽象，用于指定计算设备。生命周期起源于首次调用 `aclrtSetDevice` | Host 与 Device 为 1 比 N |
| Context | Device 的逻辑运行环境。一个 Context 属于一个唯一的 Device | Context 与 Device 为 N 比 1 |
| Stream | Device 上的执行流，同一个 Stream 中的任务执行严格保序 | Stream 与 Context 为 N 比 1 |
| Task | Device 上真正的执行体，从属于 Stream，用户编程不感知 | Task 与 Stream 为 N 比 1 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">定义</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">数量关系</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">对昇腾 AI 处理器所属设备的抽象，用于指定计算设备。生命周期起源于首次调用 <code>aclrtSetDevice</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Host 与 Device 为 1 比 N</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Context</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device 的逻辑运行环境。一个 Context 属于一个唯一的 Device</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Context 与 Device 为 N 比 1</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Stream</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device 上的执行流，同一个 Stream 中的任务执行严格保序</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Stream 与 Context 为 N 比 1</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Task</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device 上真正的执行体，从属于 Stream，用户编程不感知</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Task 与 Stream 为 N 比 1</td>
</tr>
</tbody>
</table>

关于 Context 有三条规则必须记住：

1. Context 负责管理 **Stream、Event 和 Notify** 这三类运行资源对象的生命周期，**但不包括内存**。内存不属于任何 Context。
2. 不同 Context 中的对象是完全隔离的。不同 Context 的 Stream 和 Event 完全隔离，**无法建立同步等待关系**。
3. 运行出错同样按 Context 隔离。

**与第六章的对照**：第六章的 host 侧代码只写了 `aclrtSetDevice` 与 `aclrtCreateStream`，从未出现 Context。那是因为 `aclrtSetDevice` 会隐式创建一个默认 Context，Stream 就挂在它下面。本章把这一层显式化，是因为多 Stream 与跨流同步都要求 Stream 之间同属一个 Context，否则无法建立同步等待关系。


### 2.2 线程与 Context 的关联

观察 Runtime 的接口原型可以发现，`aclrtMalloc`、`aclrtCreateStream` 这类接口都没有 Device ID 参数。这是因为它们所作用的 Device 是**从调用线程关联的 Context 中获取**的。由此产生四条规则：

1. 主机侧的 CPU 线程要关联 Context 后，才能正确调用 Runtime 接口。
2. 一个线程在同一时刻只能关联一个 Context。
3. 同一进程内的 Context 对所有线程可见，线程可以通过 `aclrtGetCurrentContext` 与 `aclrtSetCurrentContext` 切换 Context。
4. `aclrtCreateContext` 会把调用线程关联到新创建的 Context；`aclrtDestroyContext` 销毁当前线程正关联的 Context 时，会同时解除该关联，此后不带 Device 参数的 Runtime 接口无法正确调用，需要先切换到其它 Context；带 Device 参数的接口（例如 `aclrtResetDevice`、`aclrtResetDeviceForce`）不受这一关联的影响，因此本实验的 `main` 在销毁 Context 之后才复位 Device。

下面这段代码用注释标出了线程关联 Context 的变化过程，仅供理解语义，不要求编译运行。

```cpp
// 初始时，线程未关联任何 Context
aclInit(nullptr);
aclrtSetDevice(0);             // 隐式创建默认 Context，并将线程关联到默认 Context

aclrtContext ctx1 = nullptr;
aclrtContext ctx2 = nullptr;
aclrtContext current_ctx = nullptr;

aclrtCreateContext(&ctx1, 0);  // 显式创建 ctx1，线程随之关联 ctx1
aclrtGetCurrentContext(&current_ctx);   // current_ctx == ctx1
aclrtCreateContext(&ctx2, 0);  // 显式创建 ctx2，线程随之关联 ctx2
aclrtSetCurrentContext(ctx1);  // 显式切回 ctx1

aclrtDestroyContext(ctx2);     // ctx2 不是当前关联的 Context，线程关联关系不变
aclrtDestroyContext(ctx1);     // ctx1 是当前关联的 Context，销毁后线程不再关联任何 Context
aclrtResetDeviceForce(0);
```

这套规则的效果是：接口的行为取决于调用线程当前关联的上下文，而不是显式传入的参数。多线程程序中若忘记为新创建的线程设置 Context，Runtime 接口就会因为找不到 Device 而失败。在多线程下发任务的程序中会实际遇到这一情形。


### 2.3 默认 Context 与默认 Stream

设备上执行任务下发之前，必须已经存在 Context 和 Stream。二者既可以显式创建，也可以隐式创建，隐式创建出来的就是默认 Context 与默认 Stream。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 规则 | 说明 |
| --- | --- |
| 隐式创建的时机 | 调用 `aclrtSetDevice` 或 `aclrtCreateContext` 时，Runtime 会自动创建一个默认 Stream；每个 Context 拥有一个默认 Stream |
| 如何引用默认 Stream | 需要传入 Stream 参数的接口（如 `aclrtMemcpyAsync`）直接传 `nullptr` 即表示使用默认 Stream；没有 Stream 入参的接口（如 `aclrtMemcpy`）不使用默认 Stream |
| 默认 Context 的限制 | 不允许对默认 Context 执行 `aclrtGetCurrentContext`、`aclrtSetCurrentContext` 与 `aclrtDestroyContext` |
| 默认 Stream 的限制 | 默认 Stream 不能通过 `aclrtDestroyStream` 显式销毁 |
| 多线程下的共享 | 如果不同的 Host 线程使用相同的 Context，它们将共享同一个默认 Stream |
| 销毁时机 | 调用 `aclrtResetDevice` 或 `aclrtResetDeviceForce` 复位 Device 时，默认 Context 与默认 Stream 会被自动销毁 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">规则</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">隐式创建的时机</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">调用 <code>aclrtSetDevice</code> 或 <code>aclrtCreateContext</code> 时，Runtime 会自动创建一个默认 Stream；每个 Context 拥有一个默认 Stream</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">如何引用默认 Stream</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">需要传入 Stream 参数的接口（如 <code>aclrtMemcpyAsync</code>）直接传 <code>nullptr</code> 即表示使用默认 Stream；没有 Stream 入参的接口（如 <code>aclrtMemcpy</code>）不使用默认 Stream</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">默认 Context 的限制</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不允许对默认 Context 执行 <code>aclrtGetCurrentContext</code>、<code>aclrtSetCurrentContext</code> 与 <code>aclrtDestroyContext</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">默认 Stream 的限制</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">默认 Stream 不能通过 <code>aclrtDestroyStream</code> 显式销毁</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多线程下的共享</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">如果不同的 Host 线程使用相同的 Context，它们将共享同一个默认 Stream</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">销毁时机</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">调用 <code>aclrtResetDevice</code> 或 <code>aclrtResetDeviceForce</code> 复位 Device 时，默认 Context 与默认 Stream 会被自动销毁</td>
</tr>
</tbody>
</table>

《应用开发指南》对二者的适用场景给出的建议是：默认 Context 与默认 Stream 一般适用于简单应用、用户仅需要一个 Device 的计算场景；多线程应用程序建议使用显式创建的 Context 和 Stream。

本实验采用显式创建。理由有两点：一是多 Stream 程序必须显式创建 Context 与 Stream，本实验按这条通路建立范式；二是显式创建能让资源的创建与销毁在代码中一一对应，便于建立资源配对的习惯。

> ⚠️ 单个 Device 上可创建的 Stream 数量存在上限，该上限包含默认 Stream 与系统内部用于同步的 Stream。**具体数值随产品型号而不同，以对应产品的 Runtime API 参考中 `aclrtCreateStream` 的约束说明为准。**在需要大量并发流的服务化场景中需要留意这一上限。


### 2.4 设备侧不是只有 AI Core

下图列出了任务调度器可以协同调度的多种硬件加速器。需要说明的是，不同代 AI 处理器支持的硬件加速器不同，须以实际硬件用户手册为准。本实验直接用到其中两类——算子在 AI Core 上执行，主机与设备之间的数据搬运由 DMA 承担。

<img src="images/07.01_hardware_accelerators.png" alt="Runtime 协同调度的多种硬件加速器" width="700px">

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 加速器 | 承接的任务类型 | 本实验是否用到 |
| --- | --- | --- |
| AI Core（含 Vector 与 Cube） | AI Vector 任务与 AI Cube 任务 | √ `aclnnAdd` 在其上执行 |
| AI CPU | 不适合在 AI Core 上执行的 CPU 任务 | 未用到 |
| DMA | 数据拷贝任务与内存 Cache 任务 | √ 承担 `aclrtMemcpy` 的搬运 |
| DVPP | 图像与视频处理任务 | 未用到 |
| Random | 随机数生成任务 | 未用到 |
| 可编程核 | 条件算子任务 | 未用到 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">加速器</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">承接的任务类型</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验是否用到</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">AI Core（含 Vector 与 Cube）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">AI Vector 任务与 AI Cube 任务</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√ <code>aclnnAdd</code> 在其上执行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">AI CPU</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不适合在 AI Core 上执行的 CPU 任务</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">DMA</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数据拷贝任务与内存 Cache 任务</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√ 承担 <code>aclrtMemcpy</code> 的搬运</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">DVPP</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图像与视频处理任务</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Random</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">随机数生成任务</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可编程核</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">条件算子任务</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
</tbody>
</table>

图中的「硬化 Event、Notify 同步资源」说明了一件事：Event 与 Notify 并不是软件层面的标志位，而是任务调度器上的硬件资源。这也是它们的数量有上限、需要显式创建与销毁的原因。

最后把第七章的概念逐一映射到学生已经学过的内容：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 已学章节 | 已掌握的概念 | 第七章的对应概念 |
| --- | --- | --- |
| 第二章 存储层次 | 内存与 Cache 之间的带宽和延迟 | 主机内存与设备内存之间的带宽和延迟，测量方法完全相同 |
| 第四章 生产者—消费者 | 有界任务队列，入队与出队 | Stream 就是设备上的任务队列，下发即入队 |
| 第四章 屏障与条件变量 | 线程之间的等待与唤醒 | Event 是 Stream 之间的等待与唤醒 |
| 第六章 三段流水与双缓冲 | 用多重缓冲把搬运藏进计算 | 多 Stream 把主机与设备之间的传输藏进计算 |
| 第六章 固定开销 $I_0$ | 算子下发与同步的固定代价 | 单次主机—设备传输的固定开销 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">已学章节</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">已掌握的概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">第七章的对应概念</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第二章 存储层次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">内存与 Cache 之间的带宽和延迟</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">主机内存与设备内存之间的带宽和延迟，测量方法完全相同</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第四章 生产者—消费者</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">有界任务队列，入队与出队</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Stream 就是设备上的任务队列，下发即入队</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第四章 屏障与条件变量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">线程之间的等待与唤醒</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Event 是 Stream 之间的等待与唤醒</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第六章 三段流水与双缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">用多重缓冲把搬运藏进计算</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多 Stream 把主机与设备之间的传输藏进计算</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第六章 固定开销 $I_0$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子下发与同步的固定代价</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单次主机—设备传输的固定开销</td>
</tr>
</tbody>
</table>


## 3. 头文件、库文件与编译方式

### 3.1 接口分类与命名规范

acl 接口的命名风格为「acl + 接口类别缩写 + 操作动词和对象」，其中操作动词和对象均采用首字母大写。参数顺序遵循输入参数在前、输出参数在后的原则。

《应用开发指南》共列出八类接口。本章会遇到的类别如下：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 接口名前缀 | 描述 | 本实验是否用到 |
| --- | --- | --- |
| `acl` | 系统配置类接口 | √ 初始化与去初始化 |
| `aclrt` | 运行时管理类的接口 | √ Device、Context、Stream、内存管理与同步等待 |
| `aclnn` | 单算子 API，采用两段式接口形式 | √ `aclnnAdd` |
| `aclmdl` | 模型推理类的接口 | 未用到 |
| `acldvpp` | 媒体数据处理的接口 | 未用到 |
| `aclprof` | Profiling 配置类接口 | 未用到 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">接口名前缀</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">描述</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验是否用到</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>acl</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">系统配置类接口</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√ 初始化与去初始化</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrt</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">运行时管理类的接口</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√ Device、Context、Stream、内存管理与同步等待</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclnn</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单算子 API，采用两段式接口形式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√ <code>aclnnAdd</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdl</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">模型推理类的接口</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>acldvpp</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">媒体数据处理的接口</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclprof</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Profiling 配置类接口</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
</tbody>
</table>

本实验的程序会用到前三类：`aclInit`、`aclFinalize` 属于第一类，`aclrtSetDevice` 等属于第二类，`aclnnAdd` 属于第三类。


### 3.2 头文件与库文件的对应关系

acl 接口的头文件在 `${ASCEND_HOME_PATH}/include/` 目录下，库文件在 `${ASCEND_HOME_PATH}/lib64/` 目录下。此外还有一个 `${ASCEND_HOME_PATH}/devlib/` 目录，官方样例中的环境变量 `NPU_HOST_LIB` 即指向该目录，它是编译期的链接目录。

编译时的规则是：**根据程序 include 了哪些头文件，链接对应的库文件**。本实验涉及的对应关系如下：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 头文件 | 用途 | 对应的库文件 |
| --- | --- | --- |
| `acl/acl.h` | 聚合头文件，包含初始化与去初始化、Device 管理、Context 管理、Stream 管理、同步等待、内存管理等接口 | `libacl_rt.so`（旧版本为 `libascendcl.so`） |
| `aclnn/aclnn_base.h`、`aclnn/acl_meta.h` | 调用算子接口时依赖的公共数据类型，例如 `aclTensor`、`aclScalar`、`aclIntArray`；本实验由 `aclnnop/aclnn_add.h` 间接引入，源码中未直接 include | `libnnopbase.so` |
| `aclnnop/aclnn_add.h` | Add 算子的单算子 API | `libopapi_math.so`（旧版本的 `libopapi.so` 自 CANN 8.5.0 起废弃，不得使用） |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">头文件</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">用途</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">对应的库文件</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>acl/acl.h</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">聚合头文件，包含初始化与去初始化、Device 管理、Context 管理、Stream 管理、同步等待、内存管理等接口</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>libacl_rt.so</code>（旧版本为 <code>libascendcl.so</code>）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclnn/aclnn_base.h</code>、<code>aclnn/acl_meta.h</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">调用算子接口时依赖的公共数据类型，例如 <code>aclTensor</code>、<code>aclScalar</code>、<code>aclIntArray</code>；本实验由 <code>aclnnop/aclnn_add.h</code> 间接引入，源码中未直接 include</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>libnnopbase.so</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclnnop/aclnn_add.h</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Add 算子的单算子 API</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>libopapi_math.so</code>（旧版本的 <code>libopapi.so</code> 自 CANN 8.5.0 起废弃，不得使用）</td>
</tr>
</tbody>
</table>

表中第一行的 `acl/acl.h` 是聚合头文件；与 `libacl_rt.so` 对应的行是 `acl/acl_rt.h`，其用途一列的文字与上表第一行完全相同。本程序为省事直接 include 聚合头文件，但只用到其中的 Runtime 接口，因此仍然只链接 `libacl_rt.so`。

> ⚠️ 编译 acl 接口程序时，请按照 include 的头文件依赖对应的库文件。**如果引用多余的库文件（例如 `libascendcl.a`），可能导致版本功能异常或后续版本升级时存在兼容性问题。**

关于库文件，有两条版本演进需要说明，它们是本实验最容易出错的地方：

1. **`libascendcl.so` 正在按功能拆分。** 旧版本中初始化、Device、Context、Stream、内存、模型管理、单算子执行等能力统一由 `libascendcl.so` 提供，后续版本这种方式会废弃，建议改用拆分后的 `libacl_rt.so`、`libacl_mdl.so`、`libacl_op_executor.so`、`libmsprofiler.so` 等。
2. **全量算子总库 `libopapi.so` 自 CANN 8.5.0 版本开始废弃**，应改用按类别拆分后的 `libopapi_math.so`、`libopapi_nn.so`、`libopapi_cv.so`、`libopapi_transformer.so`。其中 `libopapi_math.so` 是其余三类的公共前置依赖，使用 NN、CV、Transformer 类算子时必须一并链接。

由于不同 CANN 版本的库文件名不同，把库名写死在编译命令里是不可移植的做法。本实验不写死库名，而是在下一节从本机实际存在的文件中探测。


## 4. 环境准备与检查

先把 CANN 的环境变量导入 Jupyter 进程，并创建代码目录。**这一步保证 `g++` 能找到 acl 的头文件与库文件**——Jupyter 内核继承的是启动时的环境，若 CANN 变量是在内核启动后才 source 的，这里必须重新导入一次。


In [ ]:
!mkdir -p src_skeleton

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

if os.environ.get("ASCEND_HOME_PATH"):
    print("🎉 环境变量导入完成，ASCEND_HOME_PATH =", os.environ["ASCEND_HOME_PATH"])
else:
    print("⚠️  导入后 ASCEND_HOME_PATH 仍为空，请确认 CANN 已安装且 set_env.sh 路径正确")


下面做**环境自检**：确认平台、CANN 安装路径、g++ 与 CMake 版本、以及 NPU 设备状态。若这一步不通过，本实验的编译与运行都无法进行，请先解决环境问题。


In [ ]:
import os, platform, shutil, subprocess, sys

print("=" * 62)
print(" 一、平台信息")
print("=" * 62)
print("操作系统   :", platform.system(), platform.release())
print("处理器架构 :", platform.machine())
print("Python    :", sys.version.split()[0])

print()
print("=" * 62)
print(" 二、CANN 与编译器")
print("=" * 62)
ascend_home = os.environ.get("ASCEND_HOME_PATH", "")
print("ASCEND_HOME_PATH :", ascend_home or "⚠️  未设置")
gxx = shutil.which("g++")
print("g++              :", gxx or "⚠️  未找到")
if gxx:
    v = subprocess.run(["g++", "--version"], capture_output=True, text=True)
    print("g++ 版本         :", v.stdout.splitlines()[0])
cmake = shutil.which("cmake")
print("cmake            :", cmake or "⚠️  未找到")
if cmake:
    v = subprocess.run(["cmake", "--version"], capture_output=True, text=True)
    print("cmake 版本       :", v.stdout.splitlines()[0])
print("npu-smi          :", shutil.which("npu-smi") or "⚠️  未找到")

print()
print("=" * 62)
print(" 三、NPU 设备")
print("=" * 62)
NPU_SMI_MAX_LINES = 30
if shutil.which("npu-smi"):
    proc = subprocess.run(["npu-smi", "info"], capture_output=True, text=True)
    text = proc.stdout if proc.returncode == 0 else proc.stderr
    lines = text.rstrip().splitlines()
    print("\n".join(lines[:NPU_SMI_MAX_LINES]))
    if len(lines) > NPU_SMI_MAX_LINES:
        print("...（占用进程较多，此处只显示前 %d 行）" % NPU_SMI_MAX_LINES)
else:
    print("未检测到 npu-smi，无法查询设备状态。")

print()
if ascend_home and gxx:
    print("✅ 环境就绪，可以开始实验。")
else:
    print("⚠️  环境不完整。请重新运行上一个单元格；仍不通过则在终端执行")
    print("    source $ASCEND_TOOLKIT_HOME/set_env.sh")
    print("    随后重启 Notebook 内核。")


### 4.1 探测本机实际可用的库

按 §3.2 的说明，库文件名随 CANN 版本演进。下面的代码在 `lib64` 与 `devlib` 两个目录中按优先级查找三类库文件，探测结果保存在 `ACL_LIBDIRS`、`ACL_LIBS` 与 `ACL_RT_LIB` 三个变量中，供后面的编译命令直接使用。

同时确认 `aclnnop/aclnn_add.h` 存在。若该头文件缺失，说明 CANN ops 算子包未安装或安装不完整，需要先补齐算子包再继续。


In [ ]:
import os

ascend_home = os.environ.get("ASCEND_HOME_PATH", "")
lib_dirs = [
    path
    for path in (f"{ascend_home}/lib64", f"{ascend_home}/devlib")
    if os.path.isdir(path)
]

print("ASCEND_HOME_PATH :", ascend_home)
print("可用的库目录     :", lib_dirs)
print()


# 按优先级在库目录中查找库，返回第一个存在的库名（不含 lib 前缀与 .so 后缀）
def find_lib(candidates):
    for name in candidates:
        for lib_dir in lib_dirs:
            if os.path.exists(os.path.join(lib_dir, f"lib{name}.so")):
                return name
    return None


requirements = [
    ("Runtime", ["acl_rt", "ascendcl"]),
    ("算子公共数据类型", ["nnopbase"]),
    ("Math 类算子", ["opapi_math"]),
]


# 中文字符占两列，按显示宽度补空格，避免下面几行输出错位
def pad(text, width):
    shown = sum(2 if ord(ch) > 0x2E80 else 1 for ch in text)
    return text + " " * max(0, width - shown)


selected = {}
for purpose, candidates in requirements:
    name = find_lib(candidates)
    print(pad(purpose, 22), "候选", candidates, "->", name)
    if name is not None:
        selected[purpose] = name

missing = [purpose for purpose, _ in requirements if purpose not in selected]
if missing:
    print("⚠️  未探测到：" + "、".join(missing) + "，后续编译会失败")

ACL_LIBDIRS = ["-L" + path for path in lib_dirs]
ACL_LIBS = ["-l" + selected[purpose] for purpose, _ in requirements if purpose in selected]
# 只包含 acl/acl.h 的程序仅需链接 Runtime 库，按用途取出，单独保存一份供第 8 节使用
ACL_RT_LIB = ["-l" + selected["Runtime"]] if "Runtime" in selected else []

header = os.path.join(ascend_home, "include", "aclnnop", "aclnn_add.h")
print()
print("aclnn_add.h 存在 :", os.path.exists(header))
print("ACL_LIBDIRS      :", " ".join(ACL_LIBDIRS))
print("ACL_LIBS         :", " ".join(ACL_LIBS))
print("ACL_RT_LIB       :", " ".join(ACL_RT_LIB))


## 5. 程序实现：七步框架

本实验只有**一个源文件** `src_skeleton/acl_skeleton.cpp`。与第六章不同，它是纯粹的主机侧 C++ 程序，不含任何设备侧核函数——设备上执行的是 CANN 已经编译好的内置算子。

第六章的 `.asc` 文件中，核函数入口必须定义在算子类之后，因此需要先把入口暂存到临时文件、最后再回填。本章的程序是普通的 C++ 源文件，只要保证被调用的函数定义在调用点之前，自顶向下顺序书写即可，**不需要暂存与回填**。

《应用开发指南》给出了单算子 API 调用的官方流程图，本实验的七步框架就是它的具体化：

<img src="images/07.01_aclnn_flow.png" alt="单算子 API 调用流程" width="400">

单算子 API 调用流程：虚线框内是单算子调用本身，框外的初始化、运行时资源申请、运行时资源释放与去初始化，是任何 acl 应用都要写的部分。它与本实验这七步的对照关系是：初始化 → 步骤一，运行时资源申请 → 步骤二，申请内存与传输数据 → 步骤三，计算 workspace 大小并执行算子 → 步骤四，同步等待与结果回拷 → 步骤五，释放内存与运行时资源释放 → 步骤六，去初始化 → 步骤七。

图中只有传输数据一格标为可选步骤：如果需要将 Host 上数据传输到 Device，才调用 `aclrtMemcpy` 或 `aclrtMemcpyAsync`；本实验的输入在主机侧生成，因此这一步是必需的。

文件分七段写入，每段前面都有一小节讲解。七段与七步并非一一对应：前三段是头文件、环境规格查询与辅助函数，不对应任何步骤；第七段的 `main` 同时承担步骤一与步骤七。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 顺序 | 命令 | 内容 | 对应的七步 |
| --- | --- | --- | --- |
| 1 | `%%writefile` | 头文件、常量与错误检查宏 | — |
| 2 | `-a` 追加 | 运行环境规格查询函数 | — |
| 3 | `-a` 追加 | 形状、步长与张量构造辅助函数 | — |
| 4 | `-a` 追加 | `RunVectorAdd`：创建 Context 与 Stream、准备输入 | 步骤二、步骤三 |
| 5 | `-a` 追加 | `RunVectorAdd`：调用算子、同步、取回结果并校验 | 步骤四、步骤五 |
| 6 | `-a` 追加 | `RunVectorAdd`：逆序释放资源 | 步骤六 |
| 7 | `-a` 追加 | `main`：初始化、指定 Device、去初始化 | 步骤一、步骤七 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">顺序</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">命令</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">对应的七步</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>%%writefile</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">头文件、常量与错误检查宏</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">运行环境规格查询函数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">形状、步长与张量构造辅助函数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>RunVectorAdd</code>：创建 Context 与 Stream、准备输入</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">步骤二、步骤三</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">5</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>RunVectorAdd</code>：调用算子、同步、取回结果并校验</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">步骤四、步骤五</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">6</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>RunVectorAdd</code>：逆序释放资源</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">步骤六</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">7</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>main</code>：初始化、指定 Device、去初始化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">步骤一、步骤七</td>
</tr>
</tbody>
</table>

> ⚠️ 第一个代码单元格使用 `%%writefile`（**覆盖创建**），其后六个使用 `%%writefile -a`（**追加**）。修改代码后需要从第一个单元格开始按顺序重新执行，否则文件内容会重复或缺失。

### 5.1 头文件、常量与错误检查宏

程序包含两组头文件。`acl/acl.h` 是聚合头文件，包含初始化、Device 管理、Context 管理、Stream 管理、同步等待、内存管理等接口；`aclnnop/aclnn_add.h` 是 Add 算子的单算子 API 头文件。

`ACL_CHECK` 宏承担两件事：判断返回码，以及在失败时取出错误描述。`aclGetRecentErrMsg` 取出的是本线程最近一次失败的错误描述，**其具体行为以 Runtime API 参考为准**。需要留意的是，取出一次并不等于把已有记录清空：§8.2 的附属程序会连续触发两次失败，请比对第二条 `msg=` 里的时间戳，看它是否同时含有上一条已经打印过的记录——若含有，说明错误描述是按时间顺序累积的。无论是否累积，每次调用失败都应立即取出并打印，否则后面取到的可能不是本次失败的描述。返回码的分类规则与该接口的使用规则见 §8.1。

宏中把返回值统一转换为 `int` 再比较，是因为运行时接口返回 `aclError`、算子接口返回 `aclnnStatus`，两者都是整型别名，转成 `int` 后可以用同一个宏处理。另外 `aclGetRecentErrMsg` 在获取失败时会返回空指针，直接送进 `printf` 的 `%s` 是未定义行为，因此宏中做了判空。


In [ ]:
%%writefile src_skeleton/acl_skeleton.cpp
/**
 * Parallel Computing, Chapter 7, Lab 1: Application Framework and Runtime
 * Resources
 *
 * This program computes out = self + alpha * other on an Ascend AI
 * Processor and prints the key specifications of the runtime environment.
 * Its purpose is to establish one complete application path that verifies:
 *   (1) the CANN environment variables, headers and libraries are set up
 *       correctly;
 *   (2) memory allocation, transfer and synchronization between host and
 *       device work as expected;
 *   (3) a built-in operator can be submitted and executed correctly.
 *
 * This file is a pure host-side C++ program. Section 6 builds it with a
 * single g++ command; section 7 rebuilds the same file with CMake.
 */
#include <cmath>    // std::fabs, std::fmax
#include <cstdint>  // int32_t, int64_t
#include <cstdio>   // std::printf, std::fprintf
#include <vector>   // std::vector

#include "acl/acl.h"            // Runtime resource management APIs
#include "aclnnop/aclnn_add.h"  // Single-operator API of Add

namespace {

constexpr int32_t kDeviceId = 0;
constexpr int64_t kVectorLength = 8;
constexpr float kAlphaValue = 1.0f;
constexpr double kRelTolerance = 1e-6;

}  // namespace

// Checks the return code of an acl API. On failure it prints the API name,
// the return code and the error message, then returns immediately.
// aclGetRecentErrMsg returns the most recent error message of the calling
// thread. Retrieving it right after every failure keeps the message that
// belongs to this call apart from the ones left by earlier calls.
#define ACL_CHECK(expr)                                                     \
  do {                                                                      \
    const int acl_ret = static_cast<int>(expr);                             \
    if (acl_ret != ACL_SUCCESS) {                                           \
      const char* err_msg = aclGetRecentErrMsg();                           \
      std::fprintf(stderr, "[ERR] api=%s code=%d msg=%s\n", #expr, acl_ret, \
                   (err_msg == nullptr) ? "(no message)" : err_msg);        \
      return acl_ret;                                                       \
    }                                                                       \
  } while (0)


### 5.2 运行环境规格查询

这一段的作用是把本机的关键规格打印出来并留档备查，同时验证运行时接口可以正常工作。

`aclrtGetSocName` 返回昇腾 AI 处理器的版本字符串。凡是需要填写处理器型号的场合（例如 ATC 工具的 `--soc_version` 参数），取值都以这里打印的字符串为准；先打印出来记录下来，可以避免凭记忆填错型号。

`aclrtGetDeviceInfo` 按属性查询 Device 信息，属性由 `aclrtDevAttr` 枚举给出。其中 AI Core 数量、Cube Core 数量、Vector Core 数量正是第六章反复引用的核数规格，本实验可以用它与第六章的硬件规格参照表相互印证。部分属性在特定产品型号上不受支持，接口会返回非零值，因此 `PrintDeviceAttr` 只提示不中断，并在失败分支取出错误描述丢弃，避免它干扰后续调用的判读。请对照本机输出确认哪几项属性给出了数值、哪几项打印了不受支持的提示。

`aclrtGetMemInfo` 查询设备内存的空闲大小与总大小，调用前必须已经指定 Device。


In [ ]:
%%writefile -a src_skeleton/acl_skeleton.cpp

// Queries and prints one device attribute. Some attributes are unsupported
// on certain product models; the API then returns a non-zero value and this
// function only reports it instead of aborting the program.
void PrintDeviceAttr(uint32_t device_id, aclrtDevAttr attr, const char* name) {
  int64_t value = 0;
  const aclError ret = aclrtGetDeviceInfo(device_id, attr, &value);
  if (ret == ACL_SUCCESS) {
    std::printf("[INFO] %-24s = %lld\n", name, static_cast<long long>(value));
    return;
  }
  std::printf("[INFO] %-24s = (unsupported on this model, code=%d)\n", name,
              static_cast<int>(ret));
  // Retrieve and discard the message so it does not pile up on later calls
  (void)aclGetRecentErrMsg();
}

// Prints the key specifications of the runtime environment.
// A device must have been set before this function is called.
int PrintEnvironmentInfo(int32_t device_id) {
  const char* soc_name = aclrtGetSocName();
  std::printf("[INFO] %-24s = %s\n", "soc_name",
              (soc_name == nullptr) ? "(unknown)" : soc_name);

  uint32_t device_count = 0;
  ACL_CHECK(aclrtGetDeviceCount(&device_count));
  std::printf("[INFO] %-24s = %u\n", "device_count", device_count);

  aclrtRunMode run_mode = ACL_HOST;
  ACL_CHECK(aclrtGetRunMode(&run_mode));
  std::printf("[INFO] %-24s = %s\n", "run_mode",
              (run_mode == ACL_HOST) ? "ACL_HOST" : "ACL_DEVICE");

  const uint32_t id = static_cast<uint32_t>(device_id);
  PrintDeviceAttr(id, ACL_DEV_ATTR_AICORE_CORE_NUM, "ai_core_num");
  PrintDeviceAttr(id, ACL_DEV_ATTR_CUBE_CORE_NUM, "cube_core_num");
  PrintDeviceAttr(id, ACL_DEV_ATTR_VECTOR_CORE_NUM, "vector_core_num");
  PrintDeviceAttr(id, ACL_DEV_ATTR_AICPU_CORE_NUM, "ai_cpu_num");
  PrintDeviceAttr(id, ACL_DEV_ATTR_L2_CACHE_SIZE, "l2_cache_bytes");

  size_t free_bytes = 0;
  size_t total_bytes = 0;
  ACL_CHECK(aclrtGetMemInfo(ACL_MEM_HUGE, &free_bytes, &total_bytes));
  constexpr double kBytesPerGb = 1024.0 * 1024.0 * 1024.0;
  std::printf("[INFO] %-24s = %.2f GB / %.2f GB\n", "huge_mem_free/total",
              static_cast<double>(free_bytes) / kBytesPerGb,
              static_cast<double>(total_bytes) / kBytesPerGb);
  return ACL_SUCCESS;
}


### 5.3 形状、步长与张量构造

单算子 API 用 `aclTensor` 描述一个张量。`aclCreateTensor` 的参数分为三组：第一组（前三个参数）是**视图形状**与数据类型，即算子看到的逻辑形状；第二组（第四、五个参数）是**步长与偏移**，描述视图如何映射到底层存储；第三组（第六至第九个参数）是**存储格式、存储形状与设备内存地址**。

这种把视图与存储分开描述的设计，使得切片、转置、广播等操作可以不复制数据。本实验的张量是最简单的连续一维张量，视图形状与存储形状相同、偏移为零、步长为连续布局。

有两点需要特别注意。第一，**步长以元素个数为单位，不是字节**；最后一维的步长恒为 1，其余各维由右向左依次累乘。第二，`aclCreateTensor` **只描述内存，本身不持有数据，也不负责搬运**；设备内存必须先由 `aclrtMalloc` 申请、由 `aclrtMemcpy` 填好，再把地址交给它。

> ⚠️ `aclCreateTensor` 失败时返回空指针而不是错误码，因此必须判空。


In [ ]:
%%writefile -a src_skeleton/acl_skeleton.cpp

// Returns the total number of elements of a tensor.
int64_t GetShapeSize(const std::vector<int64_t>& shape) {
  int64_t shape_size = 1;
  for (const int64_t dim : shape) {
    shape_size *= dim;
  }
  return shape_size;
}

// Computes the strides of a contiguous tensor. Strides are counted in
// elements, not in bytes. The stride of the last dimension is always 1 and
// the remaining ones are accumulated from right to left.
void ComputeStrides(const std::vector<int64_t>& shape,
                    std::vector<int64_t>* strides) {
  strides->assign(shape.size(), 1);
  for (int64_t i = static_cast<int64_t>(shape.size()) - 2; i >= 0; --i) {
    (*strides)[i] = shape[i + 1] * (*strides)[i + 1];
  }
}

// Allocates device memory, copies the host data over, and wraps it in an
// aclTensor descriptor. An aclTensor only describes the shape and layout of
// a memory block; it neither owns the data nor performs the transfer.
int CreateAclTensor(const std::vector<float>& host_data,
                    const std::vector<int64_t>& shape, void** device_addr,
                    aclTensor** tensor) {
  const size_t byte_size =
      static_cast<size_t>(GetShapeSize(shape)) * sizeof(float);
  ACL_CHECK(aclrtMalloc(device_addr, byte_size, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMemcpy(*device_addr, byte_size, host_data.data(), byte_size,
                        ACL_MEMCPY_HOST_TO_DEVICE));

  std::vector<int64_t> strides;
  ComputeStrides(shape, &strides);
  *tensor =
      aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT, strides.data(), 0,
                      ACL_FORMAT_ND, shape.data(), shape.size(), *device_addr);
  if (*tensor == nullptr) {
    std::fprintf(stderr,
                 "[ERR] api=aclCreateTensor code=- msg=returned nullptr\n");
    return ACL_ERROR_INVALID_PARAM;
  }
  return ACL_SUCCESS;
}


### 5.4 步骤二与步骤三：创建 Context 与 Stream，准备输入

从这里开始进入七步框架的主体。

**步骤二**显式创建 Context 与 Stream。`aclrtCreateStream` 不接受优先级参数；需要指定优先级时应改用 `aclrtCreateStreamWithConfig`，可设置的优先级范围由 `aclrtDeviceGetStreamPriorityRange` 查询。

**步骤三**准备输入数据。两个输入向量在主机内存里生成，随后由 `CreateAclTensor` 搬到设备上。输出张量本可以只调用 `aclrtMalloc` 申请内存，这里复用同一个函数、用一段全零数据初始化，一是为了让代码保持简短，二是可以确保结果不会与上一次运行留下的残留数据混淆——`aclrtMalloc` 申请到的内存不会被清零。

`alpha` 是一个标量系数，用 `aclCreateScalar` 构造。它与 `aclCreateTensor` 一样，失败时返回空指针。


In [ ]:
%%writefile -a src_skeleton/acl_skeleton.cpp

// Carries out steps 2 to 6 of the seven-step framework: create the context
// and the stream, prepare the inputs, launch the operator, fetch the result
// and release the resources.
int RunVectorAdd(int32_t device_id) {
  // Step 2: create the context and the stream explicitly.
  aclrtContext context = nullptr;
  ACL_CHECK(aclrtCreateContext(&context, device_id));
  aclrtStream stream = nullptr;
  ACL_CHECK(aclrtCreateStream(&stream));

  // Step 3: generate the input data in host memory.
  const std::vector<int64_t> shape{kVectorLength};
  std::vector<float> self_host(kVectorLength);
  std::vector<float> other_host(kVectorLength);
  for (int64_t i = 0; i < kVectorLength; ++i) {
    self_host[i] = static_cast<float>(i + 1);
    other_host[i] = 0.5f * static_cast<float>(i + 1);
  }

  // Step 3 (continued): copy the inputs to the device and build the input
  // and output descriptors of the operator.
  void* self_device = nullptr;
  void* other_device = nullptr;
  void* out_device = nullptr;
  aclTensor* self = nullptr;
  aclTensor* other = nullptr;
  aclTensor* out = nullptr;
  ACL_CHECK(CreateAclTensor(self_host, shape, &self_device, &self));
  ACL_CHECK(CreateAclTensor(other_host, shape, &other_device, &other));

  // The output side needs no valid input data, but aclrtMalloc does not zero
  // the memory it returns. Initializing it with zeros keeps leftover values
  // from a previous run from being mistaken for a result.
  const std::vector<float> out_host_init(kVectorLength, 0.0f);
  ACL_CHECK(CreateAclTensor(out_host_init, shape, &out_device, &out));

  float alpha_value = kAlphaValue;
  aclScalar* alpha = aclCreateScalar(&alpha_value, ACL_FLOAT);
  if (alpha == nullptr) {
    std::fprintf(stderr,
                 "[ERR] api=aclCreateScalar code=- msg=returned nullptr\n");
    return ACL_ERROR_INVALID_PARAM;
  }


### 5.5 步骤四与步骤五：调用算子、同步、取回结果

**步骤四**调用 Add 算子。CANN 内置算子采用两段式接口：

1. 第一段 `aclnnAddGetWorkspaceSize` 完成入参校验、动态形状场景下的输出形状推导、数据切块（Tiling），并算出执行该算子所需的 workspace 内存大小。**这一段全部在主机侧完成，不下发任何任务。**
2. 第二段 `aclnnAdd` 才真正把核函数下发到 Stream 上执行。

两段式接口的详细语义与设计动机见《应用开发指南》，有一点须记住：**`workspaceSize` 可能为 0**，因此申请与释放 workspace 都要先做判断，避免出错。

**步骤五**取回结果。第二段接口只是把任务下发到 Stream，返回成功仅表示下发成功，不代表任务已经执行完成。取结果之前必须调用 `aclrtSynchronizeStream` 显式同步。

校验采用相对误差而非逐位比对。本实验的输入与结果都是可以精确表示的二进制小数，逐位比对也能通过，但一旦换成随机数据，浮点计算在设备与主机上的结果不必位级一致，逐位比对必然失败。因此从第一个实验起就采用相对误差与阈值的写法。


In [ ]:
%%writefile -a src_skeleton/acl_skeleton.cpp

  // Step 4: launch the Add operator. Built-in CANN operators use a two-phase
  // interface: the first phase performs parameter checking, output shape
  // inference and tiling and reports the workspace size, while only the
  // second phase submits the kernel to the stream.
  uint64_t workspace_size = 0;
  aclOpExecutor* executor = nullptr;
  ACL_CHECK(aclnnAddGetWorkspaceSize(self, other, alpha, out, &workspace_size,
                                     &executor));
  std::printf("[INFO] %-24s = %llu\n", "workspace_size",
              static_cast<unsigned long long>(workspace_size));

  // workspace_size may be zero, so it must be checked before allocating.
  void* workspace = nullptr;
  if (workspace_size > 0) {
    ACL_CHECK(
        aclrtMalloc(&workspace, workspace_size, ACL_MEM_MALLOC_HUGE_FIRST));
  }
  ACL_CHECK(aclnnAdd(workspace, workspace_size, executor, stream));

  // Success of the second phase only means the task was submitted; an
  // explicit synchronization is required before reading the result.
  ACL_CHECK(aclrtSynchronizeStream(stream));

  // Step 5: copy the result back to the host and verify it.
  const size_t out_bytes = static_cast<size_t>(kVectorLength) * sizeof(float);
  std::vector<float> out_host(kVectorLength, 0.0f);
  ACL_CHECK(aclrtMemcpy(out_host.data(), out_bytes, out_device, out_bytes,
                        ACL_MEMCPY_DEVICE_TO_HOST));

  double max_rel_err = 0.0;
  for (int64_t i = 0; i < kVectorLength; ++i) {
    const double golden =
        static_cast<double>(self_host[i]) +
        static_cast<double>(kAlphaValue) * static_cast<double>(other_host[i]);
    const double denom = std::fmax(std::fabs(golden), 1e-12);
    const double rel_err =
        std::fabs(static_cast<double>(out_host[i]) - golden) / denom;
    max_rel_err = std::fmax(max_rel_err, rel_err);
    std::printf("[INFO] out[%lld] = %.4f, golden = %.4f\n",
                static_cast<long long>(i), out_host[i], golden);
  }
  std::printf("[VERIFY] max_rel_err=%.6e result=%s\n", max_rel_err,
              (max_rel_err <= kRelTolerance) ? "PASS" : "FAIL");


### 5.6 步骤六：逆序释放资源

释放的顺序与创建的顺序相反。具体到本段代码，有两条更细的规则：

1. **先销毁描述符，再释放描述符指向的内存。** `aclDestroyTensor` 销毁的只是描述符对象，它并不释放 `aclrtMalloc` 申请的设备内存；反过来，如果先 `aclrtFree` 再 `aclDestroyTensor`，描述符将指向一块已经释放的内存。
2. **先销毁 Stream，再销毁 Context。** Stream 的生命周期由 Context 管理，销毁 Context 时其下的 Stream 也会被回收，但显式创建的资源应当显式销毁，让代码中的创建与销毁一一对应。另外，`aclrtDestroyStream` 在 Stream 上有未完成的任务时会等待任务完成后再销毁；本程序在步骤五中已经同步过，销毁时不会再等待。

> ⚠️ 这段代码有一个必须指出的问题：如果前面任何一步失败，`ACL_CHECK` 会直接返回，此时已经申请的设备内存不会被释放。对于一个跑完就退出的教学程序，这不会造成实际影响；但在长时间运行的服务中，这就是内存泄漏。解决办法是用 RAII 把每一类资源封装成对象，由析构函数负责释放。**RAII 是工程实践中的通行做法，本章不展开；本实验直接使用接口，目的是先把每一对创建与销毁完整写出来。**


In [ ]:
%%writefile -a src_skeleton/acl_skeleton.cpp

  // Step 6: release the resources in the reverse order of creation.
  // Destroy the descriptors first, then free the device memory they point
  // to, and destroy the stream and the context last.
  ACL_CHECK(aclDestroyTensor(self));
  ACL_CHECK(aclDestroyTensor(other));
  ACL_CHECK(aclDestroyTensor(out));
  ACL_CHECK(aclDestroyScalar(alpha));

  if (workspace != nullptr) {
    ACL_CHECK(aclrtFree(workspace));
  }
  ACL_CHECK(aclrtFree(self_device));
  ACL_CHECK(aclrtFree(other_device));
  ACL_CHECK(aclrtFree(out_device));

  ACL_CHECK(aclrtDestroyStream(stream));
  ACL_CHECK(aclrtDestroyContext(context));
  return ACL_SUCCESS;
}


### 5.7 步骤一与步骤七：初始化与去初始化

`main` 函数只做三件事：初始化、调用业务函数、去初始化。

**步骤一**中，`aclInit` 必须先于其它 acl 接口调用，否则可能导致后续系统内部资源初始化出错，进而导致其它业务异常。参数传 `nullptr` 表示采用默认配置；传入一个 json 配置文件路径时，可以在不修改代码的情况下开启 Dump 或 Profiling 采集。随后的 `aclrtSetDevice` 指定本进程使用的 Device，并隐式创建默认 Context 与默认 Stream。

**步骤七**中，`aclrtResetDevice` 复位 Device、释放其上的资源；`aclFinalize` 实现系统去初始化，释放进程内 acl 接口使用的相关资源。复位 Device 有两个接口：`aclrtResetDeviceForce`，`aclrtResetDevice`。本实验是单进程单线程程序、资源已在步骤六中逐一显式销毁，因此选用 `aclrtResetDevice`。**两个接口的差别，以及 `aclrtResetDevice` 是否维护引用计数，以 Runtime API 参考为准。**

注意 `PrintEnvironmentInfo` 与 `RunVectorAdd` 的返回值都只被记录下来，没有用 `ACL_CHECK` 包住：`ACL_CHECK` 的宏体是 `return acl_ret;`，一旦用在这里，失败时就会跳过复位与去初始化，落进 §8.4 表最后一行所说的情形。一旦 `aclrtSetDevice` 成功，复位与去初始化就必须在每一条路径上执行，最后才根据记录下来的返回码决定退出码。


In [ ]:
%%writefile -a src_skeleton/acl_skeleton.cpp

int main() {
  std::printf("[INFO] acl_skeleton start\n");

  // Step 1: initialize the system and set the device used by this process.
  // aclInit must be called before any other acl API; passing nullptr
  // selects the default configuration.
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(kDeviceId));

  // Print the key specifications of the runtime environment for later labs.
  // Neither this call nor RunVectorAdd may return early: once aclrtSetDevice
  // has succeeded, the device must be reset and the system finalized on every
  // path, so both only record their return code here.
  int ret = PrintEnvironmentInfo(kDeviceId);
  if (ret == ACL_SUCCESS) {
    ret = RunVectorAdd(kDeviceId);
  }

  // Step 7: reset the device and finalize, whether or not the work above
  // succeeded.
  ACL_CHECK(aclrtResetDevice(kDeviceId));
  ACL_CHECK(aclFinalize());

  std::printf("[RESULT] %s\n", (ret == ACL_SUCCESS) ? "PASS" : "FAILED");
  std::printf("[INFO] acl_skeleton finished\n");
  return (ret == ACL_SUCCESS) ? 0 : ret;
}


## 6. 用 g++ 编译并运行

主机侧的 C++ 程序用系统自带的 `g++` 编译即可，不需要 `bisheng`。编译命令由五部分构成：源文件、语言标准与优化选项、头文件目录、库目录与库名、输出文件名。其中库目录与库名来自 §4.1 的探测结果。

`-Wall` 不能省略。本章的程序涉及大量指针与类型转换，编译告警常常先于运行期错误暴露问题。


In [ ]:
import os, subprocess

SRC = "src_skeleton/acl_skeleton.cpp"
EXE = "src_skeleton/acl_skeleton"

# g++ [源文件] -I[头文件目录] [库目录] [库名] -o [可执行文件]
cmd = (
    ["g++", SRC, "-std=c++17", "-O2", "-Wall"]
    + ["-I" + os.environ.get("ASCEND_HOME_PATH", "") + "/include"]
    + ACL_LIBDIRS
    + ACL_LIBS
    + ["-o", EXE]
)
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)


编译成功后执行。程序的输出分为三段：以 `[INFO]` 开头的环境规格与逐元素结果、以 `[VERIFY]` 开头的校验结论、以 `[RESULT]` 开头的总体结论。这套记录行格式是本章统一的约定；做性能测量的程序另有 `[PERF]` 与 `[STAGE]` 两类记录行，本实验不做性能测量，因此不出现。


In [ ]:
import subprocess

proc = subprocess.run(["./src_skeleton/acl_skeleton"], capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("返回码:", proc.returncode)
    print(proc.stderr)


## 7. 另一种构建方式：CMake 工程

上一节用一条 `g++` 命令完成了编译，简洁直接。当工程规模变大——源文件不止一个、需要链接更多的库、需要按平台或架构条件编译——通常会引入构建系统来管理这些配置。本节以**同一份源码**为例演示 CMake 的写法。

CMake 配置有三个要点：

1. **安装路径**：优先使用 `set_env.sh` 导出的 `ASCEND_HOME_PATH`，未设置时回退到默认安装路径并给出告警（该告警出现在 configure 阶段，被重定向进 `build/cmake.log`）。这样同一份 `CMakeLists.txt` 可以在不同安装位置的环境中复用。
2. **库目录**：同时加入 `lib64` 与 `devlib`，后者是编译期的链接目录（见 §3.2）。
3. **库名不写死**：与 §4.1 同理，文件中先写占位符 `ACL_LIBS_PLACEHOLDER`，再由下一个单元格替换成探测到的库名。


In [ ]:
%%writefile src_skeleton/CMakeLists.txt
cmake_minimum_required(VERSION 3.14)
project(ACL_SKELETON LANGUAGES CXX)

add_compile_options(-std=c++17 -O2 -Wall)
set(CMAKE_RUNTIME_OUTPUT_DIRECTORY "${CMAKE_CURRENT_SOURCE_DIR}/bin")

# Prefer ASCEND_HOME_PATH exported by set_env.sh; fall back to the default
# installation path when it is not set
if(DEFINED ENV{ASCEND_HOME_PATH})
  set(ASCEND_PATH $ENV{ASCEND_HOME_PATH})
else()
  message(WARNING "ASCEND_HOME_PATH not found, run: source set_env.sh")
  set(ASCEND_PATH "/usr/local/Ascend/cann")
endif()

include_directories(${ASCEND_PATH}/include ${ASCEND_PATH}/include/aclnn)

# devlib is the link-time library directory (NPU_HOST_LIB points here in the
# official samples); at run time the libraries under lib64 are loaded
link_directories(${ASCEND_PATH}/lib64 ${ASCEND_PATH}/devlib)

add_executable(acl_skeleton_cmake acl_skeleton.cpp)
target_link_libraries(acl_skeleton_cmake ACL_LIBS_PLACEHOLDER)


下面把占位符替换成 §4.1 探测到的库名，并把替换后的那一行打印出来核对。


In [ ]:
from pathlib import Path

cmake_path = Path("src_skeleton/CMakeLists.txt")
libs = " ".join(flag[2:] for flag in ACL_LIBS)
text = cmake_path.read_text(encoding="utf-8").replace("ACL_LIBS_PLACEHOLDER", libs)
cmake_path.write_text(text, encoding="utf-8")

for line in text.splitlines():
    if "target_link_libraries" in line:
        print(line)


执行 cmake 构建。

> ⚠️ **Jupyter 中的一个常见问题**：`!` 开头的每一行都是**独立的子 shell**，上一行的 `cd` 对下一行不生效。因此必须用 `&&` 把所有命令串成**一条**，并用反斜杠 `\` 续行。

其中 `-DCMAKE_SKIP_RPATH=TRUE` 表示不把库路径写入可执行文件，运行时依赖 `set_env.sh` 设置的 `LD_LIBRARY_PATH`。

configure 与编译的完整日志分别写在 `src_skeleton/build/cmake.log` 与 `src_skeleton/build/make.log` 中，任一步失败时下面的单元格会自动打印对应日志的末尾若干行。


In [ ]:
!cd src_skeleton && \
 rm -rf build && mkdir -p build && cd build && \
 { cmake .. -DCMAKE_SKIP_RPATH=TRUE > cmake.log 2>&1 || \
     { echo "❌ configure 失败，cmake.log 末尾："; tail -20 cmake.log; exit 1; }; } && \
 { make > make.log 2>&1 || \
     { echo "❌ 编译失败，make.log 末尾："; tail -20 make.log; exit 1; }; } && \
 tail -3 make.log && \
 echo "--- 运行 CMake 构建出的可执行文件 ---" && ../bin/acl_skeleton_cmake


### 两种方式的对照

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 维度 | `g++` 单行命令 | CMake 工程 |
| --- | --- | --- |
| 配置量 | 一行命令，参数直观可见 | 一个 `CMakeLists.txt` 加一次 configure |
| 多源文件 | 需手工列出全部 `.cpp` | `add_executable` 一次列全，增量编译由 make 管理 |
| 库依赖管理 | 库名与路径写在命令行里 | `link_directories` 与 `target_link_libraries` 集中管理 |
| 条件编译 | 需要手工改命令或写 shell 脚本 | `if()` 按平台、架构分支配置 |
| 适用场景 | 单文件、依赖简单的验证性程序 | 多文件、需对外交付或长期维护的工程 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">维度</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"><code>g++</code> 单行命令</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">CMake 工程</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">配置量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一行命令，参数直观可见</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一个 <code>CMakeLists.txt</code> 加一次 configure</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多源文件</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">需手工列出全部 <code>.cpp</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>add_executable</code> 一次列全，增量编译由 make 管理</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">库依赖管理</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">库名与路径写在命令行里</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>link_directories</code> 与 <code>target_link_libraries</code> 集中管理</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">条件编译</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">需要手工改命令或写 shell 脚本</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>if()</code> 按平台、架构分支配置</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">适用场景</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单文件、依赖简单的验证性程序</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多文件、需对外交付或长期维护的工程</td>
</tr>
</tbody>
</table>

本章其余实验都是单文件、库依赖简单的程序，一律用 `g++` 直接编译；这里引入 CMake，是为了对照官方样例的构建方式，并说明工程规模变大之后为什么需要构建系统。**两种方式产出的可执行文件功能等价**：上面两次运行的输出在环境规格、逐元素结果与校验结论上应当逐行一致，请对照本机的两段输出核对。


## 8. 补充：返回码、错误定位与资源配对

### 8.1 返回码的分类规则

acl 接口的返回码可以按首位数字判断成因的大类：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 返回码 | 成因 | 处理方向 |
| --- | --- | --- |
| `1xxxxx` | 环境异常或代码逻辑错误 | 可以通过优化环境或代码逻辑解决 |
| `2xxxxx` | 资源不足，或接口、参数与当前硬件不匹配 | 可以通过合理使用资源解决 |
| `3xxxxx` | 业务功能异常，例如队列满、队列空 | 检查业务逻辑 |
| `5xxxxx` | 软硬件内部异常，其中 `500000` 表示无法识别的错误 | 用户无法解决，需收集日志联系技术支持 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">返回码</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">成因</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">处理方向</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>1xxxxx</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">环境异常或代码逻辑错误</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可以通过优化环境或代码逻辑解决</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>2xxxxx</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">资源不足，或接口、参数与当前硬件不匹配</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可以通过合理使用资源解决</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>3xxxxx</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">业务功能异常，例如队列满、队列空</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">检查业务逻辑</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>5xxxxx</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">软硬件内部异常，其中 <code>500000</code> 表示无法识别的错误</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">用户无法解决，需收集日志联系技术支持</td>
</tr>
</tbody>
</table>

仅有返回码往往不足以定位问题，还需要错误描述。`aclGetRecentErrMsg` 返回本进程或本线程中其它 acl 接口调用失败时的错误描述，默认为线程级别。使用它有三条规则：

1. **每次接口调用失败都应该立即取一次。** 不立即取，之后取到的可能是更早那次失败留下的描述。
2. **取到的字符串未必只包含本次失败的记录。** 请在 §8.2 的输出中按时间戳核对：若第二条 `msg=` 里同时出现了上一条已经打印过的记录，说明错误描述是按时间顺序累积的，判读时应以最新的时间戳为准。
3. **获取失败时返回空指针**，必须判空后再使用；返回指针的有效期以 Runtime API 参考为准，稳妥的做法是取出后立即打印，不要长期保存。

### 8.2 一个只演示失败场景的附属程序

下面写一个附属程序，有意触发三种失败场景。与主程序不同，这个程序不会在失败时中断，而是把每一步的返回码与错误描述都打印出来。


In [ ]:
%%writefile src_skeleton/acl_error_demo.cpp
/**
 * Parallel Computing, Chapter 7, Lab 1, companion program:
 * retrieving return codes and error messages
 *
 * This program deliberately triggers three failure cases in order to
 * demonstrate the use of aclGetRecentErrMsg. The program itself never
 * aborts: it prints the return code and the error message of every call.
 */
#include <cstdint>
#include <cstdio>

#include "acl/acl.h"

namespace {

constexpr int32_t kDeviceId = 0;

// Prints the result of one API call, retrieving the error message on
// failure.
void ReportStatus(const char* api, int ret) {
  if (ret == ACL_SUCCESS) {
    std::printf("[OK ] api=%-32s code=%d\n", api, ret);
    return;
  }
  const char* err_msg = aclGetRecentErrMsg();
  std::printf("[ERR] api=%-32s code=%d msg=%s\n", api, ret,
              (err_msg == nullptr) ? "(no message)" : err_msg);
}

}  // namespace

int main() {
  ReportStatus("aclInit", aclInit(nullptr));

  // Case 1: query the current device before any device has been set, so
  // the call cannot succeed.
  int32_t current_device = -1;
  ReportStatus("aclrtGetDevice (no device set)",
               aclrtGetDevice(&current_device));

  // Case 2: repeated initialization. Read the printed code and classify it
  // with the table in section 8.1; the symbolic name of this code is not
  // documented in the material used by this course.
  ReportStatus("aclInit (repeated)", aclInit(nullptr));

  ReportStatus("aclrtSetDevice", aclrtSetDevice(kDeviceId));

  // Case 3: allocate zero bytes of device memory. The runtime rejects it;
  // the reason is spelled out in the message it prints.
  void* device_ptr = nullptr;
  ReportStatus("aclrtMalloc (size = 0)",
               aclrtMalloc(&device_ptr, 0, ACL_MEM_MALLOC_HUGE_FIRST));

  // Two aclInit calls were made above and only one aclFinalize is issued
  // here; compare its return code with the ones printed earlier.
  ReportStatus("aclrtResetDevice", aclrtResetDevice(kDeviceId));
  ReportStatus("aclFinalize", aclFinalize());
  return 0;
}


该程序只包含 `acl/acl.h`，因此按 §3.2 的规则**只链接 Runtime 库，不链接算子库**。这正是「按 include 的头文件链接对应的库文件」这条规则的直接应用，也是本实验唯一一处使用 `ACL_RT_LIB` 而非 `ACL_LIBS` 的地方。


In [ ]:
import os, subprocess

cmd = (
    ["g++", "src_skeleton/acl_error_demo.cpp", "-std=c++17", "-O2", "-Wall"]
    + ["-I" + os.environ["ASCEND_HOME_PATH"] + "/include"]
    + ACL_LIBDIRS
    + ACL_RT_LIB
    + ["-o", "src_skeleton/acl_error_demo"]
)
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

if proc.returncode == 0:
    print("✅ 编译成功\n")
    print(
        subprocess.run(
            ["./src_skeleton/acl_error_demo"], capture_output=True, text=True
        ).stdout
    )
else:
    print("❌ 编译失败（返回码 %d）" % proc.returncode)


### 8.3 资源配对自检表

第七章涉及的资源种类远多于第六章，资源泄漏与释放顺序错误也是本章最容易出现的故障。下表列出 acl 应用中常见的资源配对关系，每写完一个 acl 程序都应对照检查一遍。标记 √ 的行是本实验已经用到的部分；其余各行本实验未用到，列在这里是因为它们同属一套配对规则，在编写更复杂的程序时同样适用。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 创建 | 销毁 | 顺序要求 | 本实验 |
| --- | --- | --- | --- |
| `aclInit` | `aclFinalize` | 最先创建、最后销毁；多次 `aclInit` 只需一次 `aclFinalize` | √ |
| `aclrtSetDevice` | `aclrtResetDevice` | 与 `aclrtSetDevice` 配对使用；复位会销毁默认 Context 与默认 Stream | √ |
| `aclrtCreateContext` | `aclrtDestroyContext` | 销毁当前线程关联的 Context 会解除关联 | √ |
| `aclrtCreateStream` | `aclrtDestroyStream` | 有未完成任务时会等待任务完成后再销毁；默认 Stream 不能显式销毁 | √ |
| `aclrtMalloc` | `aclrtFree` | 与描述符绑定时，先销毁描述符再释放内存；workspace 一类没有描述符的内存直接释放 | √ |
| `aclCreateTensor` | `aclDestroyTensor` | 先销毁描述符，再 `aclrtFree` 数据内存 | √ |
| `aclCreateScalar` | `aclDestroyScalar` | 同上 | √ |
| `aclrtMallocHost` | `aclrtFreeHost` | 只能释放 `aclrtMallocHost` 申请的内存 | 未用到 |
| `aclrtCreateEventExWithFlag` | `aclrtDestroyEvent` | Event 先于 Stream 销毁 | 未用到 |
| `aclmdlCreateDesc` | `aclmdlDestroyDesc` | `aclmdlUnload` 在前 | 未用到 |
| `aclmdlCreateDataset` | `aclmdlDestroyDataset` | 先逐个销毁其中的 `aclDataBuffer` | 未用到 |
| `aclCreateDataBuffer` | `aclDestroyDataBuffer` | 输出侧先取地址、`aclrtFree`，再销毁 | 未用到 |
| `aclrtSubscribeReport` | `aclrtUnSubscribeReport` | 注销后再 `pthread_join` | 未用到 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">创建</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">销毁</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">顺序要求</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclInit</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclFinalize</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">最先创建、最后销毁；多次 <code>aclInit</code> 只需一次 <code>aclFinalize</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSetDevice</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtResetDevice</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与 <code>aclrtSetDevice</code> 配对使用；复位会销毁默认 Context 与默认 Stream</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtCreateContext</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtDestroyContext</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">销毁当前线程关联的 Context 会解除关联</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtCreateStream</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtDestroyStream</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">有未完成任务时会等待任务完成后再销毁；默认 Stream 不能显式销毁</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMalloc</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtFree</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与描述符绑定时，先销毁描述符再释放内存；workspace 一类没有描述符的内存直接释放</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclCreateTensor</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclDestroyTensor</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">先销毁描述符，再 <code>aclrtFree</code> 数据内存</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclCreateScalar</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclDestroyScalar</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同上</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">√</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMallocHost</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtFreeHost</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">只能释放 <code>aclrtMallocHost</code> 申请的内存</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtCreateEventExWithFlag</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtDestroyEvent</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Event 先于 Stream 销毁</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlCreateDesc</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlDestroyDesc</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlUnload</code> 在前</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlCreateDataset</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlDestroyDataset</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">先逐个销毁其中的 <code>aclDataBuffer</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclCreateDataBuffer</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclDestroyDataBuffer</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出侧先取地址、<code>aclrtFree</code>，再销毁</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSubscribeReport</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtUnSubscribeReport</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">注销后再 <code>pthread_join</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未用到</td>
</tr>
</tbody>
</table>

复位 Device 之前，应先把该 Device 上显式创建的 Event、Stream 与 Context 依次销毁，顺序是先 Event 与 Stream、再 Context、最后复位 Device，与创建顺序相反。**不按此顺序执行的具体后果，官方文档未作说明。**

### 8.4 常见故障速查

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 现象 | 原因 | 处理 |
| --- | --- | --- |
| 编译报错找不到 `acl/acl.h` | 未导入 CANN 环境变量，`ASCEND_HOME_PATH` 为空 | 重跑第 4 节的环境准备单元格 |
| 编译报错找不到 `aclnnop/aclnn_add.h` | CANN ops 算子包未安装 | 安装算子包后重跑 §4.1 的探测单元格 |
| 链接报错 `undefined reference to aclnnAdd` | 算子库未链接，或链接了已废弃的 `libopapi.so` | 检查 §4.1 打印的 `ACL_LIBS` 是否包含算子库 |
| 运行报错找不到 `.so` | 新开的终端未 source `set_env.sh`，`LD_LIBRARY_PATH` 未设置 | Notebook 中重跑环境准备单元格；终端中重新 source |
| `aclrtSetDevice` 返回失败 | Device 上的资源已被其它进程占满 | 用 `npu-smi info` 查看占用情况；注意被占用不等于必然失败，须结合返回码与错误描述判断 |
| 结果全为 0 | 忘记 `aclrtSynchronizeStream`，或回拷方向写反 | 检查同步调用与 `aclrtMemcpy` 的源、目的地址 |
| `aclrtMalloc` 返回参数非法类错误码 | 申请的 size 为 0 | size 不能为 0；本机的实际码值与错误描述见 §8.2 场景三的输出 |
| 进程退出时报错 | 未调用 `aclFinalize` | 确保去初始化在进程退出前完成 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">现象</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">原因</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">处理</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译报错找不到 <code>acl/acl.h</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未导入 CANN 环境变量，<code>ASCEND_HOME_PATH</code> 为空</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">重跑第 4 节的环境准备单元格</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译报错找不到 <code>aclnnop/aclnn_add.h</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CANN ops 算子包未安装</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">安装算子包后重跑 §4.1 的探测单元格</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">链接报错 <code>undefined reference to aclnnAdd</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子库未链接，或链接了已废弃的 <code>libopapi.so</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">检查 §4.1 打印的 <code>ACL_LIBS</code> 是否包含算子库</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">运行报错找不到 <code>.so</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">新开的终端未 source <code>set_env.sh</code>，<code>LD_LIBRARY_PATH</code> 未设置</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Notebook 中重跑环境准备单元格；终端中重新 source</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSetDevice</code> 返回失败</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device 上的资源已被其它进程占满</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">用 <code>npu-smi info</code> 查看占用情况；注意被占用不等于必然失败，须结合返回码与错误描述判断</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">结果全为 0</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">忘记 <code>aclrtSynchronizeStream</code>，或回拷方向写反</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">检查同步调用与 <code>aclrtMemcpy</code> 的源、目的地址</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMalloc</code> 返回参数非法类错误码</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">申请的 size 为 0</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">size 不能为 0；本机的实际码值与错误描述见 §8.2 场景三的输出</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">进程退出时报错</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">未调用 <code>aclFinalize</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">确保去初始化在进程退出前完成</td>
</tr>
</tbody>
</table>


## 9. 结果分析

> 以下结论针对规律与判据，不针对某一次运行的具体数值。请用本机的输出逐条核对。

**① 环境规格应当与第六章的参照表一致。**

`ai_core_num`、`cube_core_num`、`vector_core_num` 三项与第六章的硬件规格表描述的是同一块硬件。第六章曾指出，核数、块数与 `blockDim` 三者等价，且都以 AIV 计数，这里的 `vector_core_num` 就是该计数的来源。若二者对不上，应先核对 `soc_name` 是否与第六章使用的型号相同——同一机型上的多块芯片规格相同，改 `kDeviceId` 并不会改变这三项。核数之外，`l2_cache_bytes` 与 `huge_mem_free/total` 也应纳入对照：这两项在不同机型上并不相同，比核数更能反映本机与参照表是否同一机型。

**② `soc_name` 是编译期与转换期都要用到的型号标识。**

请把 §4 中 `npu-smi info` 的 Name 列与本程序打印的 `soc_name` 放在一起比较，两者的取值来源并不相同。凡是需要填写处理器型号的场合，以 `$ASCEND_HOME_PATH/compiler/data/platform_config/` 目录下的配置文件名为准——该目录下有哪些型号，本机就支持哪些型号，这是可以直接查证的依据。型号填错的后果在第六章已经出现过：算子包全程编译通过、不报任何错，只在调用时报不支持该 opType。在本实验就把型号打印出来并记录下来，是规避这类问题的第一步。

**③ `run_mode` 决定主机侧数据要不要显式拷贝。**

`ACL_HOST` 表示应用运行在主机侧，此时主机内存与设备内存是两块独立的内存，输入数据必须经 `aclrtMemcpy` 传到设备；`ACL_DEVICE` 表示软件栈运行在设备侧，无需传输数据或在设备内传输（《应用开发指南》2.9.3.2 的示例代码注释：「如果运行模式为 ACL_DEVICE……无需传输图片数据或在 Device 内传输数据；否则，需要调用内存复制接口将数据传输到 Device」）。常见的 `if (runMode == ACL_HOST)` 分支正是为这两种情形而写。请按本机打印的 `run_mode` 判断本机落在哪一侧；本实验无论落在哪一侧，都按显式拷贝的写法执行，因此结果相同。

**④ `workspace_size` 由第一段接口算出，不是固定值。**

它由 `aclnnAddGetWorkspaceSize` 根据算子类型、输入形状与切分策略算出。请读本机输出中的 `workspace_size` 一行。正因为它的取值不固定——为 0 与不为 0 两种情况都合法——程序中的 `if (workspace_size > 0)` 判断是必需的，而不是防御性冗余：`aclrtMalloc` 申请 0 字节会失败，§8.2 的场景三演示了这一点。

**⑤ `max_rel_err` 的取值可以由输入直接推出。**

本实验的输入为 1 至 8 与它们的一半，结果为 1.5 至 12.0，全部可以用 `float` 精确表示，因此设备与主机算出的结果位级一致，相对误差为零。这不是需要靠运行才能知道的结论，而是由数据本身决定的性质。一旦换成随机数据或引入超越函数，相对误差就不再为零，届时阈值的选取本身就成为一个需要论证的问题。

**⑥ 返回码与错误描述要互相印证。**

未指定 Device 就查询、重复初始化、申请 0 字节内存，成因都是代码逻辑问题，按 §8.1 的分类规则应当落在第一类，请对照本机打印的三个返回码确认。这三次失败中，请特别看重复初始化那一次：它的返回码指向重复初始化，而随之取到的错误描述讲的却是另一件事。定位问题时返回码与错误描述要互相印证，不能只看其中之一，也不能假定错误描述一定对应本次调用——这正是 §8.1 第 1 条要求每次失败都立即取一次错误描述的原因。

**⑦ 库探测的回退路径本实验并未真正检验。**

§4.1 按新库名优先、旧库名兜底的顺序查找。请看它打印的三行：若三行命中的都是候选列表中的第一个名字，说明本机装的是拆分后的新库，兜底分支没有被走到。也就是说，§3.2 讲的版本演进成立，但探测机制处理旧库名的那条路径，在本机上并没有被检验过。

---

### 🎓 结论

本实验建立了两条基本认识：**其一，主机与设备之间的一切数据往来都必须显式编写——申请两侧内存、显式拷贝、显式同步，三者缺一不可，异步接口返回成功只表示任务下发成功；其二，每一步申请的资源都必须有配对的释放，且释放顺序与创建顺序相反——先销毁描述符再释放内存，先销毁 Stream 再销毁 Context。** 后一条决定了 acl 程序的代码组织形态，也是 §8.3 那张资源配对自检表存在的理由。


## 10. 🔧 动手练习

请修改代码、重新编译并运行，观察结果的变化（建议先独立完成，再继续阅读后续内容）。

> **提示**：修改源码后需要重新执行 `%%writefile` 单元格。由于 §5.1 为覆盖写、其后六个为追加写，**须从 5.1 开始按顺序重新执行**，否则文件内容会重复或缺失。

1. **补全资源释放**。把 §5.6 的释放段全部删去，只保留 `return ACL_SUCCESS;` 与右花括号，然后按先销毁描述符、再释放内存、最后销毁 Stream 与 Context 的顺序自行补全。补全后重新编译运行，输出应与第 6 节完全一致。*思考*：为什么 `workspace` 的释放需要先判空，而三块输入输出内存不需要？

2. **改用锁页内存管理主机数据**。把 `kVectorLength` 从 8 改为 1024，并把主机侧的 `std::vector<float>` 替换为 `aclrtMallocHost` 申请的内存。要求：申请后立即用 `std::memset` 把主机侧内存清零（`aclrtMallocHost` 申请到的内存不保证已清零；设备侧内存的清零用 `aclrtMemset`，其适用侧与约束以 Runtime API 参考为准）；在释放段补上配对的 `aclrtFreeHost`，注意它只能释放 `aclrtMallocHost` 申请的内存。*提示*：`CreateAclTensor` 接收的是 `std::vector<float>&`，需要再写一个接收裸指针的版本。本题只要求写对接口配对，不要求比较两类内存的传输性能。

3. **改用默认 Context 与默认 Stream**。去掉 `aclrtCreateContext` 与 `aclrtCreateStream`，改为依赖 `aclrtSetDevice` 隐式创建的默认 Context 与默认 Stream，需要 Stream 参数的地方一律传 `nullptr`。释放段中相应地去掉 `aclrtDestroyStream` 与 `aclrtDestroyContext`。*注意*：若误写 `aclrtDestroyStream(nullptr)`，程序会报错，请记录该返回码并与 §8.1 的分类规则对照。完成后对比两版代码，结合 §2.3 的适用场景说明本实验为什么仍然坚持显式创建。

4. **验证链接规则**。把第 6 节编译命令中的 `ACL_LIBS` 换成 `ACL_RT_LIB`（即只链接 Runtime 库），重新编译，记录报错信息。再把 §8.2 的 `acl_error_demo` 改为链接完整的 `ACL_LIBS`，观察是否能编译通过。*结论*：链接缺失的库会在链接期报错，而链接多余的库通常不会立刻报错——这正是 §3.2 那条「不要引用多余的库文件」的须知**不能靠编译器来把关**的原因。


## 11. 🤔 思考题

1. `aclInit` 必须先于其它 acl 接口调用。如果漏掉这一步直接调用 `aclrtSetDevice`，会发生什么？请把这个场景补进 §8.2 的 `acl_error_demo`（注意它必须是程序中的第一次 acl 调用），记录返回码与错误描述，再结合 §8.1 的分类规则说明判断依据。

2. 程序中 `aclrtMemcpy` 的第五个参数 `kind` 写的是 `ACL_MEMCPY_HOST_TO_DEVICE`。请说明这个枚举值对本次拷贝起什么作用；即使某个版本的实现能够从源地址与目的地址反推出方向，为什么仍然应当写出语义正确的值？（提示：从接口需要据此选择哪一条传输路径的角度考虑。）

3. 一种常见的 RAII 写法是把 `aclInit` 与 `aclFinalize` 分别放进某个全局对象的构造函数与析构函数。请说明这种写法在进程退出阶段的风险（提示：不同编译单元中静态对象的析构顺序是未定义的），并给出一种既能享受 RAII 的资源自动释放、又能避开该风险的组织方式。

4. 本实验的 `RunVectorAdd` 在任何一步失败时都会直接返回，此时已经申请的设备内存不会被释放。请指出这段代码在长时间运行的服务中会造成什么后果，并给出一种在不引入 RAII 的前提下也能保证释放的写法。*（提示：对照 §5.7 中 `main` 的处理方式——它把返回码记下来，而不是直接返回。）*

5. §2.3 指出，如果不同的 Host 线程使用相同的 Context，它们将共享同一个默认 Stream。结合第四章多线程编程的经验，说明这会给多线程程序带来什么隐患。*（提示：Stream 内的任务是保序执行的。）*

6. 第六章的核函数用 `<<<blockDim, nullptr, stream>>>` 下发，本实验的算子用 `aclnnAdd(workspace, workspace_size, executor, stream)` 下发。两者的 `stream` 参数含义是否相同？两种下发方式在编译期与运行期的行为有何差异？*（本题只要求根据 §5.5 已经给出的两段式接口语义作出初步判断，不要求查阅核函数下发接口的完整说明。）*


## 12. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| 三条编程原则 | 两侧内存独立、主机与设备异步并行、Stream 内保序而 Stream 间并行 |
| 四级抽象 | Device、Context、Stream、Task 层层从属；**Context 管理 Stream / Event / Notify，但不管理内存** |
| Context 隔离 | 不同 Context 的 Stream 与 Event 完全隔离，无法建立同步等待关系；运行出错同样按 Context 隔离 |
| 线程与 Context | 多数 Runtime 接口不带 Device ID，作用的 Device 取自调用线程关联的 Context |
| 默认 Context 与默认 Stream | 由 `aclrtSetDevice` 隐式创建；Stream 参数传 `nullptr` 即为默认 Stream；二者都不能显式销毁（《应用开发指南》2.2.1、2.5.3） |
| 七步框架 | 初始化与指定 Device、创建 Context 与 Stream、准备输入、调用算子、取回结果、释放资源、去初始化 |
| 异步语义 | 算子下发后立即返回，须 `aclrtSynchronizeStream` 确认完成 |
| 两段式接口 | 第一段在主机侧完成校验、形状推导与 Tiling 并给出 workspace 大小，第二段才下发；**`workspaceSize` 可能为 0** |
| 释放顺序 | **先销毁描述符再释放内存；先销毁 Stream 再销毁 Context**；整体顺序与创建顺序相反 |
| 链接规则 | 按 include 的头文件链接对应的库；不要引用多余的库；**库名随版本演进，应从本机探测而非写死** |
| 编译方式 | 单文件用一行 `g++`；多文件、需长期维护时改用 CMake，二者产物功能等价 |
| 错误定位 | 每次调用失败都取一次 `aclGetRecentErrMsg`；返回码首位数字即成因类别 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三条编程原则</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两侧内存独立、主机与设备异步并行、Stream 内保序而 Stream 间并行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">四级抽象</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device、Context、Stream、Task 层层从属；<strong>Context 管理 Stream / Event / Notify，但不管理内存</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Context 隔离</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不同 Context 的 Stream 与 Event 完全隔离，无法建立同步等待关系；运行出错同样按 Context 隔离</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">线程与 Context</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多数 Runtime 接口不带 Device ID，作用的 Device 取自调用线程关联的 Context</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">默认 Context 与默认 Stream</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由 <code>aclrtSetDevice</code> 隐式创建；Stream 参数传 <code>nullptr</code> 即为默认 Stream；二者都不能显式销毁（《应用开发指南》2.2.1、2.5.3）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">七步框架</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">初始化与指定 Device、创建 Context 与 Stream、准备输入、调用算子、取回结果、释放资源、去初始化</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步语义</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子下发后立即返回，须 <code>aclrtSynchronizeStream</code> 确认完成</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两段式接口</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第一段在主机侧完成校验、形状推导与 Tiling 并给出 workspace 大小，第二段才下发；<strong><code>workspaceSize</code> 可能为 0</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">释放顺序</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>先销毁描述符再释放内存；先销毁 Stream 再销毁 Context</strong>；整体顺序与创建顺序相反</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">链接规则</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">按 include 的头文件链接对应的库；不要引用多余的库；<strong>库名随版本演进，应从本机探测而非写死</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译方式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单文件用一行 <code>g++</code>；多文件、需长期维护时改用 CMake，二者产物功能等价</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">错误定位</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每次调用失败都取一次 <code>aclGetRecentErrMsg</code>；返回码首位数字即成因类别</td>
</tr>
</tbody>
</table>

### 一条贯穿本章的原则

> **在主机侧编程中，资源的申请与释放、数据的搬运与同步，全部要成对地写出来；未配对的一半会在资源受限或长时间运行时暴露为故障。**

第六章的一条贯穿原则是凡是自动发生的机制都不再自动，本章是它在资源生命周期这一维度上的延伸。第六章的资源只有 UB 缓冲与队列两类，配对关系一眼可见；acl 应用的资源有十余类，创建与销毁常常相隔上百行代码，因此需要 §8.3 那张自检表。

### 本实验没有回答的问题

本实验搬运了三块共 96 字节的数据，小到无需关心耗时，因此全文没有出现任何一个时间量。数据从主机搬到设备要花多少时间、这个时间由什么决定、怎样把搬运的代价降下来，都还没有涉及。这些问题需要先有一套可信的测量方法，而测量方法要成立，前提正是本实验做扎实的三件事：环境规格清楚、资源配对无误、错误能够定位。
